# 10 — Engagement gains from reuse

Two reuse mechanisms, measured the same way: **how much extra engagement (views, likes,
comments) does a derived video collect, relative to the original footage it came from?**

1. **Clipping gains** (PP *and* NM): source = a channel's own **RV**; derivatives = the
   Shorts / TikToks cut from it. Same channel on both sides. Edges from `gs_pp_only` /
   `gs_news_only`.
2. **Repost gains** (PP only): source = the politician's **NM appearance** (canonical NM
   RV); derivative = the politician's repost on their own channel. Edges from `gs_diff`,
   restricted to `reuse_direction == 'nm_to_pp'` (built exactly as in nb09).

**Design decisions made explicit (change them in one place, Section 1):**

- **Per-clip *and* aggregated-to-source.** One RV can spawn several clips. `pairs_*` holds
  one row per (source, derivative). `by_source_*` sums the derivatives of each source, so
  "additional views from clipping RV *x*" = (sum of its clips' views) − (RV's views).
- **Absolute gain *and* ratio.** Absolute gain (`gain_*`) is dominated by a few viral
  videos and does not average across channels; the **amplification ratio**
  (`derivative / source`) is scale-free and is what we aggregate (by **median**, since it
  is right-skewed). Both are computed; reporting defaults to the ratio.
- **Platform is not silently mixed.** A YouTube "view" and a TikTok "view" are different
  metrics, and an RV↔TikTok comparison crosses them. Every pair carries
  `source_platform` / `derivative_platform`; clipping gains are reported **split by
  derivative platform** (YT-clip vs TT-clip). Restrict to YT→YT for a clean same-metric
  comparison.

# 0. Setup & imports

In [1]:
import os
import networkx as nx

from scripts.project_config import get_party_orientation, get_politician2party

if os.path.isdir("../data/"):
    os.chdir("../")

%load_ext autoreload
%autoreload 2

from src.utils import *
from notebooks.plots_utils import *

politician2party   = get_politician2party()
party_colors       = get_parties_color_maps()
party_orientation  = get_party_orientation()


# 1. Configuration — engagement column names & options

**IMPORTANT / VERIFY.** The existing notebooks (08, 09) never touch engagement fields, so
I could not confirm from your code how views/likes/comments are named in your JSONL. The
defaults below follow the raw API conventions (YouTube API v3 `statistics.*`, TikTok
Research API `*_count`). **Run the schema check in the next cell**; if a name is wrong it
will list the columns you actually have so you can fix `ENGAGEMENT_COLS` here.

In [2]:
# --- Which raw column holds each metric, per platform. EDIT if the check below fails. ---
ENGAGEMENT_COLS = {
    'yt': {'views': 'viewCount',    'likes': 'likeCount',    'comments': 'commentCount'},
    'tt': {'views': 'view_count',   'likes': 'like_count',   'comments': 'comment_count'},
}
METRICS = ['views', 'likes', 'comments']   # drop 'comments' if too sparse; add 'shares' for TT if present

# Ratio hygiene: sources with 0 (or missing) on a metric make the ratio undefined -> dropped
# from that metric's ratio aggregation (kept for absolute gain). Set a floor if you prefer.
MIN_SOURCE_FOR_RATIO = 1     # require source metric >= this to compute a ratio
CLIP_MIN_PAIRS       = 3     # per-channel: min matched pairs to report a channel-level number


# 2. Load videos (YT + TT, both periods) and build engagement lookups

Same loading as nb08/nb09. We keep 2022 + 2024 so pairs are resolvable whichever period a video is from; filter by `publishedAt` later if you want 2024 only.

In [26]:
# ---- YouTube ----
yt_nm_2022 = pd.read_json("data/youtube/videos/news_videos_2022.jsonl", lines=True)
yt_nm_2024 = pd.read_json("data/youtube/videos/news_videos_2024.jsonl", lines=True)
yt_pp_2022 = pd.read_json("data/youtube/videos/pp_videos_2022.jsonl",   lines=True)
yt_pp_2024 = pd.read_json("data/youtube/videos/pp_videos_2024.jsonl",   lines=True)
YT_DFS = [yt_nm_2022, yt_nm_2024, yt_pp_2022, yt_pp_2024]
for _df in YT_DFS:
    _df['publishedAt'] = pd.to_datetime(_df['publishedAt']).dt.tz_localize(None)
    _df.rename(columns = {'views_2025-04':"viewCount", 'likes_2025-04':"likeCount", 'comments_2025-04':"commentCount",
                          'views_2025-01':"viewCount", 'likes_2025-01':"likeCount", 'comments_2025-01':"commentCount",
                             'views_2025-08':"viewCount", 'likes_2025-08':"likeCount", 'comments_2025-08':"commentCount"}, inplace=True)

# ---- TikTok ----
tt_nm_2022 = pd.read_json("data/tiktok/videos/news_videos_2022.jsonl", lines=True)
tt_nm_2024 = pd.read_json("data/tiktok/videos/news_videos_2024.jsonl", lines=True)
tt_pp_2022 = pd.read_json("data/tiktok/videos/pp_videos_2022.jsonl",   lines=True)
tt_pp_2024 = pd.read_json("data/tiktok/videos/pp_videos_2024.jsonl",   lines=True)
TT_DFS = [tt_nm_2022, tt_nm_2024, tt_pp_2022, tt_pp_2024]
for _df in TT_DFS:
    _df.rename(columns={'id': 'videoId', 'voice_to_text': 'transcript',
                        'create_time': 'publishedAt'}, inplace=True)
    _df['videoId'] = _df['videoId'].apply(str)
    _df['publishedAt'] = pd.to_datetime(_df['publishedAt']).dt.tz_localize(None)


In [28]:
# ---- Unified lookups (mirror nb09) ----
def get_platform_from_videoId(videoid):
    return 'yt' if len(str(videoid)) == 11 else 'tt'

videoId2publishedAt = {}
videoid2standard_name = {}
videoId2yt_type = {}     # 'RV'/'Short' for YT, 'TikTok' for TT

for _df in YT_DFS + TT_DFS:
    videoId2publishedAt.update(dict(zip(_df['videoId'], _df['publishedAt'])))
    videoid2standard_name.update(dict(zip(_df['videoId'], _df['name_standard'])))
for _df in YT_DFS:
    videoId2yt_type.update(dict(zip(_df['videoId'],
                             ['Short' if e else 'RV' for e in _df['isShort']])))
for _df in TT_DFS:
    videoId2yt_type.update({vid: 'TikTok' for vid in _df['videoId']})

# ---- videoId -> {metric: value}, coerced to numeric, per platform ----
def _engagement_lookup(dfs, plat):
    colmap = ENGAGEMENT_COLS[plat]
    out = {}
    for _df in dfs:
        sub = _df[['videoId'] + list(colmap.values())].copy()
        for m, c in colmap.items():
            sub[m] = pd.to_numeric(sub[c], errors='coerce')
        for r in sub.itertuples(index=False):
            out[getattr(r, 'videoId')] = {m: getattr(r, m) for m in colmap}
    return out

eng_yt = _engagement_lookup(YT_DFS, 'yt')
eng_tt = _engagement_lookup(TT_DFS, 'tt')

def get_engagement(videoId, metric):
    v = str(videoId)
    d = eng_yt.get(v) or eng_tt.get(v)
    return np.nan if d is None else d.get(metric, np.nan)

print(f"engagement known for {len(eng_yt)} YT + {len(eng_tt)} TT videos")


engagement known for 82061 YT + 13515 TT videos


# 3. Load graphs + channel lists (as in nb08/nb09)

In [29]:
def load_gs(filepath):
    filenames = [f for f in os.listdir(filepath) if f.endswith('.gv')]
    return {f.split('_')[0]: nx.nx_agraph.read_dot(filepath + f) for f in filenames}

gs_pp_only   = load_gs('data/networks/pp_only_2024/')
gs_news_only = load_gs('data/networks/news_only_2024/')
gs_diff      = load_gs('data/networks/diff_2024/')

nm_yt_channels = pd.read_json('data/youtube/channels/news_channels.json')
pp_yt_channels = pd.read_json('data/youtube/channels/pp_channels.json')
nm_name2orientation = dict(zip(nm_yt_channels.name_standard, nm_yt_channels.orientation))

stat_not_significant_channels = ['Gabriel Attal', 'LP', 'LR', 'RE', 'Rima Hassan', 'UDI']
yt_pp_standard_names = list(set(pp_yt_channels.name_standard) - set(stat_not_significant_channels))
yt_nm_standard_names = list(nm_yt_channels.name_standard)


# 4. Clipping gains (PP and NM)

For each channel graph, every connected component is one source RV plus the short-form
clips cut from it (same convention nb08 uses via `count_distinct_cuts`). We emit one
**pair** per (RV, clip). A component with no RV node (only a Short/TikTok, no captured
long-form) has no measurable source and is skipped — recorded in `n_components_no_rv`.

In [30]:
def clipping_pairs(gs, name_standard_list):
    """One row per (source RV, derived clip) for every channel in the list."""
    rows, skipped = [], {}
    for name in name_standard_list:
        if name not in gs:
            continue
        g = gs[name].to_undirected()
        no_rv = 0
        for comp in nx.connected_components(g):
            rv_nodes = [n for n in comp if videoId2yt_type.get(n) == 'RV']
            if not rv_nodes:
                no_rv += 1
                continue
            src = rv_nodes[0]                     # canonical source = the RV
            clips = [n for n in comp if n != src]
            for c in clips:
                rows.append({
                    'name_standard': name,
                    'party': politician2party.get(name),
                    'source': src,
                    'derivative': c,
                    'source_type': videoId2yt_type.get(src),
                    'derivative_type': videoId2yt_type.get(c),
                    'source_platform': get_platform_from_videoId(src),
                    'derivative_platform': get_platform_from_videoId(c),
                })
        skipped[name] = no_rv
    df = pd.DataFrame(rows)
    return df, skipped


def add_engagement(df, metrics=METRICS):
    """Attach source/derivative metric values, absolute gain, and amplification ratio."""
    df = df.copy()
    for m in metrics:
        df[f'src_{m}'] = df['source'].map(lambda v: get_engagement(v, m))
        df[f'der_{m}'] = df['derivative'].map(lambda v: get_engagement(v, m))
        df[f'gain_{m}'] = df[f'der_{m}'] - df[f'src_{m}']
        denom = df[f'src_{m}'].where(df[f'src_{m}'] >= MIN_SOURCE_FOR_RATIO)
        df[f'ratio_{m}'] = df[f'der_{m}'] / denom
    return df


In [31]:
pairs_pp_clip, skip_pp = clipping_pairs(gs_pp_only,   yt_pp_standard_names)
pairs_nm_clip, skip_nm = clipping_pairs(gs_news_only, yt_nm_standard_names)
pairs_pp_clip = add_engagement(pairs_pp_clip)
pairs_nm_clip = add_engagement(pairs_nm_clip)

print(f"PP clip pairs: {len(pairs_pp_clip)}  |  NM clip pairs: {len(pairs_nm_clip)}")
print("components with no RV source (PP):", sum(skip_pp.values()),
      "| (NM):", sum(skip_nm.values()))
pairs_pp_clip.head()


PP clip pairs: 839  |  NM clip pairs: 4200
components with no RV source (PP): 1262 | (NM): 4946


,name_standard,party,source,derivative,source_type,derivative_type,source_platform,derivative_platform,src_views,der_views,...,der_likes,gain_likes,ratio_likes,src_comments,der_comments,gain_comments,ratio_comments,src_pub,der_pub,elapsed_days
0,Jordan Bardella,RN,ygpHlnyS-V4,LQYZKYlbyIg,RV,RV,yt,yt,8195,48744,...,2208.0,1571.0,3.466248,138.0,443.0,305.0,3.210145,2024-04-08 10:08:42,2024-03-03 16:10:55,-35.748461
1,Jordan Bardella,RN,ygpHlnyS-V4,PE9y73jJJRg,RV,Short,yt,yt,8195,329912,...,18626.0,17989.0,29.240188,138.0,1657.0,1519.0,12.007246,2024-04-08 10:08:42,2024-04-19 10:00:30,10.994306
2,Jordan Bardella,RN,ygpHlnyS-V4,7342585614694354209,RV,TikTok,yt,tt,8195,101124,...,8187.0,7550.0,12.852433,138.0,153.0,15.0,1.108696,2024-04-08 10:08:42,2024-03-04 19:02:20,-34.629421
3,Jordan Bardella,RN,ygpHlnyS-V4,7348136699349732640,RV,TikTok,yt,tt,8195,769059,...,82498.0,81861.0,129.510204,138.0,809.0,671.0,5.862319,2024-04-08 10:08:42,2024-03-19 18:03:22,-19.670370
4,Jordan Bardella,RN,ygpHlnyS-V4,7344056865170525472,RV,TikTok,yt,tt,8195,212405,...,24645.0,24008.0,38.689168,138.0,388.0,250.0,2.811594,2024-04-08 10:08:42,2024-03-08 18:11:35,-30.664664


In [46]:
# ---- Aggregate clips to their source RV: 'additional engagement from clipping this RV' ----
def by_source(pairs, metrics=METRICS):
    g = pairs.groupby(['name_standard', 'party', 'source', 'source_platform'])
    agg = g.agg(
        n_clips=('derivative', 'size'),
        der=('derivative', lambda s: '|'.join(map(str, sorted(set(s))))),
    ).reset_index()
    for m in metrics:
        src_first = g[f'src_{m}'].first().reset_index(drop=True)
        sum_der   = g[f'der_{m}'].sum(min_count=1).reset_index(drop=True)
        agg[f'src_{m}']       = src_first
        agg[f'sum_clip_{m}']  = sum_der
        agg[f'total_gain_{m}'] = agg[f'sum_clip_{m}'] - agg[f'src_{m}']
        denom = agg[f'src_{m}'].where(agg[f'src_{m}'] >= MIN_SOURCE_FOR_RATIO)
        agg[f'amplification_{m}'] = agg[f'sum_clip_{m}'] / denom
    return agg

by_source_pp = by_source(pairs_pp_clip)
by_source_nm = by_source(pairs_nm_clip)
by_source_pp.head()


,name_standard,party,source,source_platform,n_clips,der,src_views,sum_clip_views,total_gain_views,amplification_views,src_likes,sum_clip_likes,total_gain_likes,amplification_likes,src_comments,sum_clip_comments,total_gain_comments,amplification_comments
0,EELV,EELV,AytMAraIsN0,yt,1,7385432419312209184,13969,3615,-10354,0.258787,465.0,517.0,52.0,1.111828,98.0,11.0,-87.0,0.112245
1,EELV,EELV,GnjmEcxk7Yc,yt,1,7385613061534633248,9423,1440,-7983,0.152818,324.0,130.0,-194.0,0.401235,257.0,11.0,-246.0,0.042802
2,EELV,EELV,SXBVd2WoEOY,yt,7,7389014072567631136|HbmVwqVRnjc|JsOPGIeTB1A|Vo...,14572,73739,59167,5.060321,413.0,3241.0,2828.0,7.847458,310.0,954.0,644.0,3.077419
3,EELV,EELV,T7LQXzEcsXY,yt,1,7377799124764101920,1445,983,-462,0.680277,40.0,60.0,20.0,1.500000,24.0,4.0,-20.0,0.166667
4,EELV,EELV,fj8VHEzuPGk,yt,2,7388259253515013408|Gw7QWICElZM,2504,3873,1369,1.546725,201.0,365.0,164.0,1.815920,38.0,31.0,-7.0,0.815789


### 4a. Clipping gains — summary tables

Reported as the **median amplification ratio** (derivative engagement as a multiple of the
source's own), which is the scale-free, skew-robust number. `>1` means the reuse collected
more than the original; `<1` means less. Split by **derivative platform** so YouTube-view
and TikTok-view comparisons are not blended, and the pure same-metric YT→YT case is
isolated.

In [47]:
def clipping_summary(pairs, label, metrics=METRICS):
    """Per-derivative-platform medians of the amplification ratio + per-clip absolute gain."""
    out = []
    for der_plat, sub in pairs.groupby('derivative_platform'):
        clean = 'same-metric (YT RV -> YT clip)' if der_plat == 'yt' else 'cross-metric (YT RV -> TT clip)'
        row = {'set': label, 'derivative_platform': der_plat, 'comparison': clean,
               'n_pairs': len(sub)}
        for m in metrics:
            row[f'median_ratio_{m}'] = sub[f'ratio_{m}'].median()
            row[f'median_gain_{m}']  = sub[f'gain_{m}'].median()
            row[f'share_amplified_{m}'] = (sub[f'ratio_{m}'] > 1).mean()   # fraction of clips that beat their RV
        out.append(row)
    return pd.DataFrame(out)

clip_summary = pd.concat([
    clipping_summary(pairs_pp_clip, 'PP'),
    clipping_summary(pairs_nm_clip, 'NM'),
], ignore_index=True)
clip_summary.round(3)


,set,derivative_platform,comparison,n_pairs,median_ratio_views,median_gain_views,share_amplified_views,median_ratio_likes,median_gain_likes,share_amplified_likes,median_ratio_comments,median_gain_comments,share_amplified_comments
0,PP,tt,cross-metric (YT RV -> TT clip),510,1.262,1775.0,0.547,1.343,258.5,0.524,0.378,-192.0,0.308
1,PP,yt,same-metric (YT RV -> YT clip),329,1.725,12730.0,0.669,2.294,1510.0,0.699,0.673,-67.0,0.374
2,NM,tt,cross-metric (YT RV -> TT clip),2647,1.423,1934.0,0.516,3.719,224.0,0.624,0.391,-11.0,0.287
3,NM,yt,same-metric (YT RV -> YT clip),1553,1.025,177.0,0.471,1.487,35.0,0.549,0.490,-15.0,0.304


### 4b. Clipping gains — per channel and per party

Per-channel amplification (median across that channel's clips), reported only for channels
with at least `CLIP_MIN_PAIRS` matched pairs; then averaged to party level. Restricted to
**YT→YT** here for a clean same-metric read — flip `DER_PLATFORM` to `'tt'` for the TikTok
side, or `None` for both.

In [54]:
def median_ratio_views(pairs, platform_filter=None):
    """Per-channel median of per-pair views ratio, optionally filtered by derivative platform."""
    sub = pairs if platform_filter is None else pairs[pairs['derivative_platform'] == platform_filter]
    g = sub.groupby(['name_standard', 'party'])
    out = g['ratio_views'].agg(['median', 'size']).reset_index()
    out = out[out['size'] >= CLIP_MIN_PAIRS]          # same support threshold as before
    return out.set_index(['name_standard', 'party'])['median']

# clip amplification, split by where the clip landed
yt_short = median_ratio_views(pairs_pp_clip, 'yt').rename('views_via_shorts')
tiktok   = median_ratio_views(pairs_pp_clip, 'tt').rename('views_via_tiktok')

# repost amplification (PP repost views vs NM appearance views); reposts has no
# 'derivative_platform' col, so filter on pp_platform=None => all reposts
rep_g = reposts.groupby(['name_standard', 'party'])
repost = rep_g['ratio_views'].agg(['median', 'size'])
repost = repost[repost['size'] >= CLIP_MIN_PAIRS]['median'].rename('views_via_reposts')

chan_views = (pd.concat([yt_short, tiktok, repost], axis=1)
                .reset_index())
chan_views.round(2)

,name_standard,party,views_via_shorts,views_via_tiktok,views_via_reposts
0,EELV,EELV,0.61,0.30,0.33
1,Emmanuel Macron,RE,2.60,112.90,1.26
2,François Ruffin,LFI,0.46,0.27,0.32
3,Jean-Luc Mélenchon,LFI,1.59,1.65,3.44
4,Jordan Bardella,RN,3.80,19.04,1.82
5,LFI,LFI,3.41,NaN,1.30
6,Manon Aubry,LFI,1.03,2.04,0.83
7,Manuel Bompard,LFI,1.67,0.74,2.77
8,Marine Le Pen,RN,6.72,21.43,0.61
9,Marion Maréchal,REC,0.89,1.04,1.19


In [59]:
# channels that survived into the table, per party
kept = chan_views.dropna().set_index('name_standard')['party'].to_dict()

def pair_counts(pairs, platform_filter=None):
    sub = pairs if platform_filter is None else pairs[pairs['derivative_platform'] == platform_filter]
    sub = sub[sub['name_standard'].isin(kept)]
    return sub.groupby('name_standard').size().groupby(kept).sum()  # -> per party

n_shorts  = pair_counts(pairs_pp_clip, 'yt').rename('n_shorts')
n_tiktok  = pair_counts(pairs_pp_clip, 'tt').rename('n_tiktok')
n_repost  = (reposts[reposts['name_standard'].isin(kept)]
             .groupby('name_standard').size().groupby(kept).sum().rename('n_repost'))

party_views = (chan_views.dropna()
    .groupby('party')[['views_via_shorts', 'views_via_tiktok', 'views_via_reposts']]
    .mean()
    .assign(n_channels=chan_views.dropna().groupby('party').size()))

party_views = party_views.join([n_shorts, n_tiktok, n_repost]).fillna(0)


DAGGER_THRESHOLD = 24

def fmt(val, n):
    mark = "$^{\\dagger}$" if n < DAGGER_THRESHOLD else ""
    return f"{val:.2f}{mark}"

order = ['views_via_shorts', 'n_shorts', 'views_via_tiktok', 'n_tiktok',
         'views_via_reposts', 'n_repost']
for party, r in party_views[order].iterrows():
    print(f"{party} & {fmt(r['views_via_shorts'], r['n_shorts'])} & {int(r['n_shorts'])} & "
          f"{fmt(r['views_via_tiktok'], r['n_tiktok'])} & {int(r['n_tiktok'])} & "
          f"{fmt(r['views_via_reposts'], r['n_repost'])} & {int(r['n_repost'])} \\\\")

EELV & 0.61$^{\dagger}$ & 7 & 0.30$^{\dagger}$ & 11 & 0.33$^{\dagger}$ & 18 \\
LFI & 1.39 & 128 & 1.62 & 198 & 2.19 & 156 \\
PCF & 2.32 & 96 & 0.60 & 60 & 0.98 & 64 \\
RE & 2.60$^{\dagger}$ & 4 & 112.90$^{\dagger}$ & 14 & 1.26$^{\dagger}$ & 7 \\
REC & 0.76 & 32 & 0.78 & 97 & 1.01 & 73 \\
RN & 5.26 & 53 & 20.24 & 58 & 1.21 & 78 \\


# why RE tiktok gains so high ? 

In [63]:
pairs_pp_clip[(pairs_pp_clip['party'] == 'RE') & (pairs_pp_clip['derivative_type']== 'TikTok')]

,name_standard,party,source,derivative,source_type,derivative_type,source_platform,derivative_platform,src_views,der_views,...,der_likes,gain_likes,ratio_likes,src_comments,der_comments,gain_comments,ratio_comments,src_pub,der_pub,elapsed_days
597,Emmanuel Macron,RE,fslUZZ_i4DU,7367756877209357600,RV,TikTok,yt,tt,13525,5086822,...,277685.0,277117.0,488.882042,273.0,14631.0,14358.0,53.593407,2024-05-09 07:30:09,2024-05-11 14:59:45,2.312222
598,Emmanuel Macron,RE,fslUZZ_i4DU,7368119656382582049,RV,TikTok,yt,tt,13525,2642963,...,203354.0,202786.0,358.017606,273.0,4723.0,4450.0,17.300366,2024-05-09 07:30:09,2024-05-12 14:27:43,3.289977
599,Emmanuel Macron,RE,fslUZZ_i4DU,7368095433517714720,RV,TikTok,yt,tt,13525,1740285,...,50608.0,50040.0,89.098592,273.0,3873.0,3600.0,14.186813,2024-05-09 07:30:09,2024-05-12 12:53:30,3.224549
600,Emmanuel Macron,RE,fslUZZ_i4DU,7368015406138756384,RV,TikTok,yt,tt,13525,817867,...,40049.0,39481.0,70.508803,273.0,1166.0,893.0,4.271062,2024-05-09 07:30:09,2024-05-12 07:42:58,3.008900
601,Emmanuel Macron,RE,fslUZZ_i4DU,7367788304382692640,RV,TikTok,yt,tt,13525,1313527,...,78935.0,78367.0,138.970070,273.0,2353.0,2080.0,8.619048,2024-05-09 07:30:09,2024-05-11 17:01:41,2.396898
602,Emmanuel Macron,RE,fslUZZ_i4DU,7367651453185379616,RV,TikTok,yt,tt,13525,1772156,...,139474.0,138906.0,245.552817,273.0,2188.0,1915.0,8.014652,2024-05-09 07:30:09,2024-05-11 08:10:40,2.028137
603,Emmanuel Macron,RE,fslUZZ_i4DU,7367692331706797344,RV,TikTok,yt,tt,13525,4790263,...,361178.0,360610.0,635.876761,273.0,10059.0,9786.0,36.846154,2024-05-09 07:30:09,2024-05-11 10:49:14,2.138252
604,Emmanuel Macron,RE,fslUZZ_i4DU,7367675674632490272,RV,TikTok,yt,tt,13525,2614573,...,155611.0,155043.0,273.963028,273.0,9680.0,9407.0,35.457875,2024-05-09 07:30:09,2024-05-11 09:44:36,2.093368
605,Emmanuel Macron,RE,fslUZZ_i4DU,7368046206871735584,RV,TikTok,yt,tt,13525,671209,...,27726.0,27158.0,48.813380,273.0,1175.0,902.0,4.304029,2024-05-09 07:30:09,2024-05-12 09:42:27,3.091875
607,Emmanuel Macron,RE,fslUZZ_i4DU,7368187420690500896,RV,TikTok,yt,tt,13525,1015501,...,47156.0,46588.0,83.021127,273.0,3592.0,3319.0,13.157509,2024-05-09 07:30:09,2024-05-12 18:50:27,3.472431


In [65]:
pairs_pp_clip[(pairs_pp_clip['source'] == 'fslUZZ_i4DU') & (pairs_pp_clip['derivative_type']== 'TikTok')]['der_views'].sum()

np.int64(27164655)

In [66]:
257881/31960

8.06886733416771

In [67]:
1587971/98148

16.1793515914741

In [68]:
265318/12084

21.95614035087719

In [69]:
(21 + 8 + 16) / 3

15.0